# AML Typology Injector
---
**Purpose**: Inject realistic AML typology patterns into the base transaction dataset.
**Input**: `transactions_base.csv`, `accounts.csv`, `customers.csv`
**Output**: `transactions_final.csv` with `is_aml`, `aml_typology`, `typology_group_id` labels

### Typologies Implemented
| # | Typology | Pattern |
|---|----------|--------|
| 1 | Structuring (Smurfing) | Multiple sub-threshold deposits across accounts |
| 2 | Circular Transaction Loops | A -> B -> C -> A money circles |
| 3 | Funnel Account Networks | Many-to-one concentration then rapid outflow |
| 4 | Pass-Through Transit Hubs | Receive & forward within minutes |
| 5 | Rapid Multi-Hop Layering | 8-10 hop chains within hours |


## 1 -- Load Base Data


In [ ]:
import csv, random, string, hashlib, os, json, copy
from datetime import datetime, timedelta
from collections import defaultdict, Counter

random.seed(99)

DATA_DIR = "aml_synthetic_data"

def load_csv(filename):
    path = os.path.join(DATA_DIR, filename)
    with open(path, 'r', encoding='utf-8') as f:
        return list(csv.DictReader(f))

transactions = load_csv("transactions_base.csv")
accounts = load_csv("accounts.csv")
customers = load_csv("customers.csv")

with open(os.path.join(DATA_DIR, "config.json")) as f:
    CONFIG = json.load(f)

# Index structures
acct_to_cif = {a["account_number"]: a["customer_cif"] for a in accounts}
cif_to_accts = defaultdict(list)
for a in accounts:
    if a["account_status"] == "Active":
        cif_to_accts[a["customer_cif"]].append(a["account_number"])
active_accounts = [a["account_number"] for a in accounts if a["account_status"] == "Active"]
cust_lookup = {c["customer_cif"]: c for c in customers}

# Track used timestamps per account
used_ts = defaultdict(set)
for t in transactions:
    used_ts[t["customer_account_number"]].add((t["datestamp"], t["timestamp"]))

print(f"Loaded {len(transactions):,} base transactions")
print(f"Loaded {len(accounts):,} accounts, {len(customers):,} customers")
print(f"Active accounts available: {len(active_accounts):,}")



## 2 -- Typology Injection Helpers


In [ ]:
def generate_txn_id():
    return "TXN" + ''.join(random.choices(string.ascii_uppercase + string.digits, k=16))

def safe_timestamp(account_number, date_str, preferred_hour=None):
    # Generate a unique timestamp for this account on this date
    for _ in range(3600):
        h = preferred_hour if preferred_hour is not None else random.randint(8, 22)
        m = random.randint(0, 59)
        s = random.randint(0, 59)
        ts = f"{h:02d}:{m:02d}:{s:02d}"
        key = (date_str, ts)
        if key not in used_ts[account_number]:
            used_ts[account_number].add(key)
            return ts
    return f"{random.randint(0,23):02d}:{random.randint(0,59):02d}:{random.randint(0,59):02d}"

def make_aml_txn(acct_from, acct_to, amount, date_str, hour, typology, group_id, channel="NEFT"):
    # Create a single synthetic AML transaction record
    cif_from = acct_to_cif.get(acct_from, "")
    cif_to_val = acct_to_cif.get(acct_to, "")
    cust = cust_lookup.get(cif_from, {})
    cp_cust = cust_lookup.get(cif_to_val, {})

    ts = safe_timestamp(acct_from, date_str, preferred_hour=hour)

    return {
        "transaction_id": generate_txn_id(),
        "timestamp": ts,
        "datestamp": date_str,
        "transaction_amount": round(amount, 2),
        "currency": "INR",
        "transaction_type_dr_cr": "Dr",
        "transaction_mode_channel_bank": channel,
        "cash_flag": "Y" if channel in ("Branch Cash", "ATM") else "N",
        "transaction_type_ppi": "",
        "transaction_mode_channel_ppi": "",
        "transaction_status": "Success",
        "wallet_balance_before": "",
        "wallet_balance_after": "",
        "source_of_funds_wallet": "",
        "load_instrument_type": "",
        "load_source_masked": "",
        "beneficiary_wallet_vpa": "",
        "merchant_id": "",
        "merchant_name": "",
        "mcc_code": "",
        "merchant_location": "",
        "refund_chargeback_flag": "N",
        "customer_account_number": acct_from,
        "customer_cif": cif_from,
        "counterparty_account_number": acct_to,
        "counterparty_ifsc_swift": next((a["customer_branch_ifsc"] for a in accounts if a["account_number"] == acct_to), "SBIN0000001"),
        "customer_name": cust.get("customer_name", ""),
        "counterparty_name": cp_cust.get("customer_name", ""),
        "sender_country_code": "IN",
        "receiver_country_code": "IN",
        "device_id": hashlib.sha256(acct_from.encode()).hexdigest()[:32],
        "ip_address": f"103.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}",
        "geo_city_country": f"{cust.get('city','Mumbai')}, IN",
        "gps_lat": cust.get("address_lat", ""),
        "gps_lon": cust.get("address_lon", ""),
        "browser_app_info": random.choice(CONFIG.get("browsers", ["Chrome/124.0"])),
        "session_id": hashlib.md5(f"{acct_from}{date_str}{random.random()}".encode()).hexdigest()[:24],
        "auth_method": random.choice(["OTP", "MPIN", "Biometric"]),
        "vpn_flag": "N",
        "emulator_flag": "N",
        "wallet_id": "",
        "is_aml": 1,
        "aml_typology": typology,
        "typology_group_id": group_id,
    }

def random_date_in_range():
    start = datetime.strptime(CONFIG["date_range_start"], "%Y-%m-%d")
    end = datetime.strptime(CONFIG["date_range_end"], "%Y-%m-%d")
    d = start + timedelta(days=random.randint(0, (end-start).days))
    return d.strftime("%d-%m-%Y")

def pick_n_accounts(n, exclude=None):
    # Pick n distinct active accounts, optionally excluding some
    exclude = set(exclude or [])
    pool = [a for a in active_accounts if a not in exclude]
    return random.sample(pool, min(n, len(pool)))

print("Typology helpers loaded")



## 3 -- Typology 1: Structuring (Smurfing)
**Pattern**: A person deposits amounts just below the INR 10,000 reporting threshold across multiple accounts, then consolidates into one account.


In [ ]:
structuring_txns = []
NUM_STRUCTURING_SCENARIOS = 80

for scenario_idx in range(NUM_STRUCTURING_SCENARIOS):
    group_id = f"STRUCT_{scenario_idx+1:04d}"

    target_acct = random.choice(active_accounts)
    n_sources = random.randint(3, 6)
    source_accts = pick_n_accounts(n_sources, exclude=[target_acct])

    if len(source_accts) < 3:
        continue

    date_str = random_date_in_range()
    base_hour = random.randint(9, 16)

    # Step 1: Multiple deposits just below threshold into source accounts
    for i, src in enumerate(source_accts):
        deposit_amount = random.uniform(8000, 9900)
        txn = make_aml_txn(
            acct_from=src, acct_to=src,
            amount=deposit_amount,
            date_str=date_str,
            hour=base_hour + (i % 6),
            typology="Structuring (Smurfing)",
            group_id=group_id,
            channel="Branch Cash"
        )
        txn["transaction_type_dr_cr"] = "Cr"
        txn["cash_flag"] = "Y"
        structuring_txns.append(txn)

    # Step 2: Transfer from all sources to target (next day)
    next_date = datetime.strptime(date_str, "%d-%m-%Y") + timedelta(days=random.randint(1, 3))
    next_date_str = next_date.strftime("%d-%m-%Y")

    for i, src in enumerate(source_accts):
        transfer_amount = random.uniform(7500, 9800)
        txn = make_aml_txn(
            acct_from=src, acct_to=target_acct,
            amount=transfer_amount,
            date_str=next_date_str,
            hour=random.randint(10, 18),
            typology="Structuring (Smurfing)",
            group_id=group_id,
            channel=random.choice(["NEFT", "IMPS", "UPI"])
        )
        structuring_txns.append(txn)

print(f"Structuring scenarios: {NUM_STRUCTURING_SCENARIOS}, transactions: {len(structuring_txns)}")



## 4 -- Typology 2: Circular Transaction Loops
**Pattern**: A -> B -> C -> A. Money circulates in a ring to simulate fake business activity.


In [ ]:
circular_txns = []
NUM_CIRCULAR_SCENARIOS = 60

for scenario_idx in range(NUM_CIRCULAR_SCENARIOS):
    group_id = f"CIRC_{scenario_idx+1:04d}"

    ring_size = random.randint(3, 5)
    ring_accts = pick_n_accounts(ring_size)
    if len(ring_accts) < 3:
        continue

    base_amount = random.uniform(50000, 500000)
    date_str = random_date_in_range()

    for hop in range(ring_size):
        sender = ring_accts[hop]
        receiver = ring_accts[(hop + 1) % ring_size]

        amount = base_amount * random.uniform(0.97, 1.0)
        hop_date = datetime.strptime(date_str, "%d-%m-%Y") + timedelta(days=hop)
        hop_date_str = hop_date.strftime("%d-%m-%Y")

        # Debit side (sender)
        txn = make_aml_txn(
            acct_from=sender, acct_to=receiver,
            amount=amount,
            date_str=hop_date_str,
            hour=random.randint(10, 17),
            typology="Circular Transaction Loop",
            group_id=group_id,
            channel=random.choice(["NEFT", "RTGS", "IMPS"])
        )
        circular_txns.append(txn)

        # Credit side (receiver)
        cr_txn = make_aml_txn(
            acct_from=receiver, acct_to=sender,
            amount=amount,
            date_str=hop_date_str,
            hour=random.randint(10, 17),
            typology="Circular Transaction Loop",
            group_id=group_id,
            channel=random.choice(["NEFT", "RTGS", "IMPS"])
        )
        cr_txn["transaction_type_dr_cr"] = "Cr"
        cr_txn["customer_account_number"] = receiver
        cr_txn["counterparty_account_number"] = sender
        circular_txns.append(cr_txn)

print(f"Circular loop scenarios: {NUM_CIRCULAR_SCENARIOS}, transactions: {len(circular_txns)}")



## 5 -- Typology 3: Funnel Account Networks
**Pattern**: Many accounts (15-50) send money to a single funnel account (e.g., fake NGO), which quickly sends it to a destination.


In [ ]:
funnel_txns = []
NUM_FUNNEL_SCENARIOS = 40

for scenario_idx in range(NUM_FUNNEL_SCENARIOS):
    group_id = f"FUNNEL_{scenario_idx+1:04d}"

    funnel_acct = random.choice(active_accounts)
    n_feeders = random.randint(15, 50)
    feeders = pick_n_accounts(n_feeders, exclude=[funnel_acct])
    destination = pick_n_accounts(1, exclude=[funnel_acct] + feeders)
    if not destination or len(feeders) < 10:
        continue
    destination = destination[0]

    date_str = random_date_in_range()

    # Step 1: Many small-to-medium credits into funnel
    total_funnel = 0
    for i, feeder in enumerate(feeders):
        amount = random.uniform(5000, 30000)
        total_funnel += amount
        day_offset = random.randint(0, 5)
        feed_date = datetime.strptime(date_str, "%d-%m-%Y") + timedelta(days=day_offset)
        feed_date_str = feed_date.strftime("%d-%m-%Y")

        txn = make_aml_txn(
            acct_from=feeder, acct_to=funnel_acct,
            amount=amount,
            date_str=feed_date_str,
            hour=random.randint(8, 20),
            typology="Funnel Account Network",
            group_id=group_id,
            channel=random.choice(["UPI", "IMPS", "NEFT"])
        )
        funnel_txns.append(txn)

    # Step 2: Large outflow from funnel to destination
    out_date = datetime.strptime(date_str, "%d-%m-%Y") + timedelta(days=random.randint(6, 10))
    out_date_str = out_date.strftime("%d-%m-%Y")

    n_splits = random.randint(2, 3)
    split_amounts = []
    remaining = total_funnel * 0.95
    for s in range(n_splits - 1):
        amt = remaining * random.uniform(0.3, 0.5)
        split_amounts.append(amt)
        remaining -= amt
    split_amounts.append(remaining)

    for i, amt in enumerate(split_amounts):
        out_day = out_date + timedelta(days=i)
        txn = make_aml_txn(
            acct_from=funnel_acct, acct_to=destination,
            amount=amt,
            date_str=out_day.strftime("%d-%m-%Y"),
            hour=random.randint(10, 16),
            typology="Funnel Account Network",
            group_id=group_id,
            channel="RTGS" if amt > 200000 else "NEFT"
        )
        funnel_txns.append(txn)

print(f"Funnel scenarios: {NUM_FUNNEL_SCENARIOS}, transactions: {len(funnel_txns)}")



## 6 -- Typology 4: Pass-Through Transit Hubs
**Pattern**: Account receives a large sum and forwards nearly all of it within minutes - no funds retained.


In [ ]:
passthrough_txns = []
NUM_PASSTHROUGH_SCENARIOS = 70

for scenario_idx in range(NUM_PASSTHROUGH_SCENARIOS):
    group_id = f"PASS_{scenario_idx+1:04d}"

    transit_acct = random.choice(active_accounts)
    source_acct = pick_n_accounts(1, exclude=[transit_acct])
    dest_acct = pick_n_accounts(1, exclude=[transit_acct] + source_acct)
    if not source_acct or not dest_acct:
        continue
    source_acct = source_acct[0]
    dest_acct = dest_acct[0]

    amount_in = random.uniform(200000, 2000000)
    amount_out = amount_in * random.uniform(0.96, 0.99)

    date_str = random_date_in_range()
    hour = random.randint(10, 17)

    # Inbound credit to transit account
    txn_in = make_aml_txn(
        acct_from=source_acct, acct_to=transit_acct,
        amount=amount_in,
        date_str=date_str,
        hour=hour,
        typology="Pass-Through Transit Hub",
        group_id=group_id,
        channel=random.choice(["RTGS", "NEFT"])
    )
    txn_in["transaction_type_dr_cr"] = "Cr"
    txn_in["customer_account_number"] = transit_acct
    txn_in["counterparty_account_number"] = source_acct
    passthrough_txns.append(txn_in)

    # Outbound debit from transit (within minutes - same hour or next)
    txn_out = make_aml_txn(
        acct_from=transit_acct, acct_to=dest_acct,
        amount=amount_out,
        date_str=date_str,
        hour=min(hour + random.choice([0, 0, 0, 1]), 23),
        typology="Pass-Through Transit Hub",
        group_id=group_id,
        channel=random.choice(["RTGS", "NEFT", "IMPS"])
    )
    passthrough_txns.append(txn_out)

print(f"Pass-through scenarios: {NUM_PASSTHROUGH_SCENARIOS}, transactions: {len(passthrough_txns)}")



## 7 -- Typology 5: Rapid Multi-Hop Layering
**Pattern**: Money rapidly traverses 8-10 intermediary accounts within hours to obscure origin.


In [ ]:
layering_txns = []
NUM_LAYERING_SCENARIOS = 50

for scenario_idx in range(NUM_LAYERING_SCENARIOS):
    group_id = f"LAYER_{scenario_idx+1:04d}"

    n_hops = random.randint(8, 10)
    chain = pick_n_accounts(n_hops + 1)
    if len(chain) < 9:
        continue

    base_amount = random.uniform(100000, 1000000)
    date_str = random_date_in_range()
    start_hour = random.randint(9, 14)

    for hop in range(len(chain) - 1):
        sender = chain[hop]
        receiver = chain[hop + 1]

        # Amount decreases slightly each hop
        amount = base_amount * (0.99 ** hop) * random.uniform(0.98, 1.0)

        minutes_offset = hop * random.randint(5, 30)
        hop_hour = min(start_hour + minutes_offset // 60, 23)

        day_offset = minutes_offset // (24 * 60)
        hop_date = datetime.strptime(date_str, "%d-%m-%Y") + timedelta(days=day_offset)
        hop_date_str = hop_date.strftime("%d-%m-%Y")

        txn = make_aml_txn(
            acct_from=sender, acct_to=receiver,
            amount=amount,
            date_str=hop_date_str,
            hour=hop_hour,
            typology="Rapid Multi-Hop Layering",
            group_id=group_id,
            channel=random.choice(["IMPS", "NEFT", "UPI", "RTGS"])
        )
        layering_txns.append(txn)

print(f"Layering scenarios: {NUM_LAYERING_SCENARIOS}, transactions: {len(layering_txns)}")



## 8 -- Merge & Export Final Dataset


In [ ]:
all_aml_txns = structuring_txns + circular_txns + funnel_txns + passthrough_txns + layering_txns

print("=" * 60)
print("AML Typology Injection Summary")
print("=" * 60)
print(f"  Structuring (Smurfing):       {len(structuring_txns):>6,} txns")
print(f"  Circular Transaction Loops:   {len(circular_txns):>6,} txns")
print(f"  Funnel Account Networks:      {len(funnel_txns):>6,} txns")
print(f"  Pass-Through Transit Hubs:    {len(passthrough_txns):>6,} txns")
print(f"  Rapid Multi-Hop Layering:     {len(layering_txns):>6,} txns")
print(f"  {'-'*40}")
print(f"  Total AML transactions:       {len(all_aml_txns):>6,}")
print(f"  Base (clean) transactions:    {len(transactions):>6,}")

# Merge
final_transactions = transactions + all_aml_txns
random.shuffle(final_transactions)

total = len(final_transactions)
aml_count = sum(1 for t in final_transactions if str(t.get("is_aml", "0")) == "1")
clean_count = total - aml_count

print(f"\n  Final dataset size:           {total:>6,}")
print(f"  AML-labeled:                  {aml_count:>6,} ({100*aml_count/total:.2f}%)")
print(f"  Clean-labeled:                {clean_count:>6,} ({100*clean_count/total:.2f}%)")
print("=" * 60)



## 9 -- Final Timestamp Collision Check


In [ ]:
ts_check = defaultdict(list)
for t in final_transactions:
    ts_check[t["customer_account_number"]].append((t["datestamp"], t["timestamp"]))

collision_count = 0
for acct, ts_list in ts_check.items():
    c = Counter(ts_list)
    for k, v in c.items():
        if v > 1:
            collision_count += 1

if collision_count == 0:
    print("OK: No timestamp collisions - each account has unique (date, time) pairs")
else:
    print(f"Found {collision_count} timestamp collisions - deduplicating...")
    seen = defaultdict(set)
    for t in final_transactions:
        acct = t["customer_account_number"]
        key = (t["datestamp"], t["timestamp"])
        while key in seen[acct]:
            parts = t["timestamp"].split(":")
            s = int(parts[2]) + 1
            if s >= 60:
                s = 0
                m = int(parts[1]) + 1
                if m >= 60:
                    m = 0
                    h = int(parts[0]) + 1
                    if h >= 24:
                        h = 0
                    parts[0] = f"{h:02d}"
                parts[1] = f"{m:02d}"
            parts[2] = f"{s:02d}"
            t["timestamp"] = ":".join(parts)
            key = (t["datestamp"], t["timestamp"])
        seen[acct].add(key)
    print("  OK: Collisions resolved")



## 10 -- Save Final Dataset


In [ ]:
filepath = os.path.join(DATA_DIR, "transactions_final.csv")
if final_transactions:
    keys = final_transactions[0].keys()
    with open(filepath, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(final_transactions)
    print(f"Saved: {filepath}")
    print(f"  Rows: {len(final_transactions):,}")
    print(f"  Columns: {len(keys)}")
else:
    print("WARNING: No transactions to save!")

print(f"\n-- Output Files --")
for fn in sorted(os.listdir(DATA_DIR)):
    fpath = os.path.join(DATA_DIR, fn)
    size_mb = os.path.getsize(fpath) / (1024*1024)
    print(f"  {fn:35s} {size_mb:>8.2f} MB")



## 11 -- Typology Distribution Analysis


In [ ]:
print("\n-- Typology Distribution --\n")
typ_counts = defaultdict(int)
typ_groups = defaultdict(set)
for t in final_transactions:
    typ = t.get("aml_typology", "")
    if typ:
        typ_counts[typ] += 1
        typ_groups[typ].add(t.get("typology_group_id", ""))

for typ in sorted(typ_counts.keys()):
    count = typ_counts[typ]
    groups = len(typ_groups[typ])
    print(f"  {typ:35s}  {count:>6,} txns across {groups:>4} scenarios")

print(f"\n  {'-'*55}")
total_aml = sum(typ_counts.values())
total_clean = len(final_transactions) - total_aml
print(f"  {'TOTAL AML':35s}  {total_aml:>6,}")
print(f"  {'TOTAL CLEAN':35s}  {total_clean:>6,}")
print(f"  {'GRAND TOTAL':35s}  {len(final_transactions):>6,}")

print("\nNotebook 2 complete - dataset ready for model training!")

